# Importação de bibliotecas

In [ ]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

# Preparando os dados

### Carregando os dados

In [ ]:
df_treino = pd.read_csv('./dataset/train.csv')
df_teste = pd.read_csv('./dataset/test.csv')

print(f"Total de linhas: {df_treino.shape[0]}")
print(f"Total de colunas: {df_treino.shape[1]}\n")
print("Valores nulos por coluna:")
print(df_treino.isna().sum())

### Limpando os dados

In [ ]:
df_treino['text_clean'] = df_treino['text_clean'].fillna('')
df_teste['text_clean'] = df_teste['text_clean'].fillna('')
df_treino['text_head_30'] = df_treino['text_head_30'].fillna('')
df_teste['text_head_30'] = df_teste['text_head_30'].fillna('')

### Enriquecendo os dados

In [ ]:
def extrair_features_score(df):
    df_copy = df.copy()
    df_copy['score_ARE'] = df_copy[['kw_agravo', 'kw_seguimento', 'kw_negou_seguimento', 'kw_inadmissibilidade']].fillna(0).sum(axis=1)
    df_copy['score_Acordao'] = df_copy[['kw_colegiado', 'kw_unanimidade']].fillna(0).sum(axis=1)
    df_copy['score_Despacho'] = df_copy[['kw_intime', 'kw_cite', 'kw_notifique']].fillna(0).sum(axis=1)
    df_copy['score_RE'] = df_copy[['kw_constitucional', 'kw_prequestionamento']].fillna(0).sum(axis=1)
    df_copy['score_Sentenca'] = df_copy[['kw_julgo', 'kw_procedente', 'kw_improcedente', 'kw_condeno']].fillna(0).sum(axis=1)
    return df_copy

df_treino_enriquecido = extrair_features_score(df_treino).dropna(subset=['Category']).reset_index(drop=True)
df_teste_enriquecido = extrair_features_score(df_teste)

### Vetorização de texto e redução de dimensionalidade

In [ ]:
tfidf_texto_completo = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), sublinear_tf=True)
tfidf_completo_treino = tfidf_texto_completo.fit_transform(df_treino_enriquecido['text_clean'])
tfidf_completo_teste = tfidf_texto_completo.transform(df_teste_enriquecido['text_clean'])

svd_texto_completo = TruncatedSVD(n_components=70, random_state=42)
svd_completo_treino = svd_texto_completo.fit_transform(tfidf_completo_treino)
svd_completo_teste = svd_texto_completo.transform(tfidf_completo_teste)

tfidf_inicio_texto = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), sublinear_tf=True)
tfidf_inicio_treino = tfidf_inicio_texto.fit_transform(df_treino_enriquecido['text_head_30'])
tfidf_inicio_teste = tfidf_inicio_texto.transform(df_teste_enriquecido['text_head_30'])

svd_inicio_texto = TruncatedSVD(n_components=40, random_state=42)
svd_inicio_treino = svd_inicio_texto.fit_transform(tfidf_inicio_treino)
svd_inicio_teste = svd_inicio_texto.transform(tfidf_inicio_teste)

features_numericas_treino = df_treino_enriquecido.iloc[:, 5:].fillna(0).values
features_numericas_teste = df_teste_enriquecido.iloc[:, 4:].fillna(0).values

y_treino = df_treino_enriquecido['Category'].values

X_treino_final = np.hstack((features_numericas_treino, svd_completo_treino, svd_inicio_treino))
X_teste_final = np.hstack((features_numericas_teste, svd_completo_teste, svd_inicio_teste))

# Treinamento do modelo

In [ ]:
params_finais = {
    "n_estimators": 500,
    "learning_rate": 0.07636050836101001,
    "num_leaves": 24,
    "max_depth": 4,
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1
}

classificador_final = LGBMClassifier(**params_finais)

classificador_final.fit(X_treino_final, y_treino)

# Avaliação do modelo

In [ ]:
cv_estratificada = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
historico_f1_scores = []

for fold, (idx_treino, idx_val) in enumerate(cv_estratificada.split(X_treino_final, y_treino), 1):
    X_treino_fold, y_treino_fold = X_treino_final[idx_treino], y_treino[idx_treino]
    X_validacao_fold, y_validacao_fold = X_treino_final[idx_val], y_treino[idx_val]
    
    classificador_fold = LGBMClassifier(**params_finais)
    classificador_fold.fit(X_treino_fold, y_treino_fold)
    
    previsoes_validacao = classificador_fold.predict(X_validacao_fold)
    score_f1 = f1_score(y_validacao_fold, previsoes_validacao, average='macro')
    historico_f1_scores.append(score_f1)

    print(f"Fold {fold} - F1-Score Macro: {score_f1:.4f}")

print(f"\nF1-Score Médio: {np.mean(historico_f1_scores):.4f} (+/- {np.std(historico_f1_scores):.4f})")

# Aplicando modelo no conjunto de teste

In [ ]:
previsoes_teste = classificador_final.predict(X_teste_final)

df_submissao = pd.DataFrame({
    'Id': df_teste['Id'],
    'Category': previsoes_teste
})

df_submissao.to_csv('./submission.csv', index=False)